<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.2_web_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-ollama ollama tavily-python

In [1]:
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv

_ = load_dotenv(dotenv_path=".env", override=True)

In [2]:
import os
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gemma4:31b-cloud",
    model_provider="ollama",
    base_url="https://ollama.com",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

## Without web search

In [3]:
import time
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(model=llm)

start_time = time.time()

messages = [HumanMessage(content="""
How up to date is your training knowledge?
""")]
response = agent.invoke(input={"messages": messages})

for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

================================ Human Message =================================


How up to date is your training knowledge?

================================== Ai Message ==================================

My knowledge cutoff is **January 2025**. I do not have information about events or developments that have occurred after that date unless they are provided to me within the current conversation context.
Time taken: 15.13s seconds


## Add web search tool

### Tavily Web Search

In [4]:
from tavily import TavilyClient
from typing import Dict, Any
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, domains: list) -> Dict[str, Any]:
    """
    Search the web for information
    """

    return tavily_client.search(
        query=query,
        include_domains=domains
    )

web_search.invoke(input={
"query": "List of public holidays in Malaysia for the year 2026.",
"domains": ["timeanddate.com", "officeholidays.com"]
})

{'query': 'List of public holidays in Malaysia for the year 2026.',
 'response_time': 0.83,
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.officeholidays.com/guide/malaysia/2026',
   'title': 'Holiday Guide for Malaysia in 2026 | Office Holidays',
   'content': "List of National Holidays in Malaysia in 2026 ; Isra and Miraj (in lieu). Sunday January 18th. Isra and Miraj (in lieu). Terengganu: Isra and Mi'raj marks the",
   'score': 0.99999726,
   'raw_content': None},
  {'url': 'https://www.officeholidays.com/countries/malaysia/2026',
   'title': 'National Holidays in Malaysia in 2026',
   'content': 'List of Holidays in Malaysia in 2026 ; Tuesday, Feb 03, Thaipusam (in lieu) ; Tuesday, Feb 17, Chinese New Year ; Wednesday, Feb 18, Chinese New Year Holiday',
   'score': 0.999984,
   'raw_content': None},
  {'url': 'https://www.timeanddate.com/holidays/malaysia/2026',
   'title': 'Holidays and Observances in Malaysia in 2026 - Time and Da

In [5]:
from langchain.agents import create_agent

system_prompt = """
You are a helpful assistant.
When using the web_search tool, you MUST pass:
- query: string
- domains: list of domains

Please keep the below structure.
Country code: The country code of the holiday.
Date: The date of the holiday.
Name: The name of the holiday.
"""

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt
)

In [6]:
import time
from langchain.messages import HumanMessage

start_time = time.time()

messages = [HumanMessage(content="""
List of public holidays in Malaysia for the year 2026.

Use the web_search tool with:
- query: "Malaysia public holidays 2026"
- domains: ["timeanddate.com", "officeholidays.com"]
""")]
response = agent.invoke(input={"messages": messages})

for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================


List of public holidays in Malaysia for the year 2026.

Use the web_search tool with:
- query: "Malaysia public holidays 2026"
- domains: ["timeanddate.com", "officeholidays.com"]

================================== Ai Message ==================================
Tool Calls:
  web_search (6022a8ee-5d28-48d0-b272-79364668f1ac)
 Call ID: 6022a8ee-5d28-48d0-b272-79364668f1ac
  Args:
    domains: ['timeanddate.com', 'officeholidays.com']
    query: Malaysia public holidays 2026
================================= Tool Message =================================
Name: web_search

{"query": "Malaysia public holidays 2026", "response_time": 1.17, "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.timeanddate.com/holidays/malaysia/2026", "title": "Holidays and Observances in Malaysia in 2026 - Time and Date", "content": "| 1 Jan | Thursday | New Year's Day | State Holiday | All

### Ollama Web Search

In [7]:
import os
from ollama import Client
from typing import Dict, Any
from langchain.tools import tool

client = Client(
    host="https://ollama.com/api/web_search",
    headers={"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}
)

@tool
def web_search(query: str, domains: list) -> Dict[str, Any]:
    """
    Search the web for information
    """

    domains = " OR ".join([f"site:{d}" for d in domains])
    query = f"{query} ({domains})"
    return client.web_search(query=query)

web_search.invoke(input={
"query": "List of public holidays in Malaysia for the year 2026.",
"domains": ["timeanddate.com", "officeholidays.com"]
})

WebSearchResponse(results=[WebSearchResult(content="Holidays and Observances in Malaysia in 2026\n\nShowing: all Public holidaysPublic holidays and non-working daysHolidays and some observancesHolidays (incl. some local) and observancesHolidays (incl. all local) and observancesAll holidays/observances/religious eventsCustom – choose holidays... For: 200020012002200320042005200620072008200920102011201220132014201520162017201820192020202120222023202420252026Today20272028202920302031203220332034203520362037203820392040 Select AllClear AllReset to Default Federal/National Holidays (15)Common Local Holidays (14)Local holidays (28)Important observances (4)Seasons (4)Major Hinduism (0)\n\n## Holidays and Observances in Malaysia in 2026\n\n| Date | Name | Type | Details |\n| --- | --- | --- | --- |\n| Jan 1 | Thursday | New Year's Day | State Holiday | All except JHR, KDH, KTN, PLS, TRG |\n| Jan 14 | Wednesday | Birthday of Yang di-Pertuan Besar | State Holiday | NSN |\n| Jan 17 | Saturday | I

In [8]:
from langchain.agents import create_agent

system_prompt = """
You are a helpful assistant.
When using the web_search tool, you MUST pass:
- query: string
- domains: list of domains

Please keep the below structure.
Country code: The country code of the holiday.
Date: The date of the holiday.
Name: The name of the holiday.
"""

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=system_prompt
)

In [9]:
import time
from langchain.messages import HumanMessage

start_time = time.time()

messages = [HumanMessage(content="""
List of public holidays in Malaysia for the year 2026.

Use the web_search tool with:
- query: "Malaysia public holidays 2026"
- domains: ["timeanddate.com", "officeholidays.com"]
""")]
response = agent.invoke(input={"messages": messages})

for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

================================ Human Message =================================


List of public holidays in Malaysia for the year 2026.

Use the web_search tool with:
- query: "Malaysia public holidays 2026"
- domains: ["timeanddate.com", "officeholidays.com"]

================================== Ai Message ==================================
Tool Calls:
  web_search (42f12e20-556f-4075-b0e4-735cb3602f07)
 Call ID: 42f12e20-556f-4075-b0e4-735cb3602f07
  Args:
    domains: ['timeanddate.com', 'officeholidays.com']
    query: Malaysia public holidays 2026
================================= Tool Message =================================
Name: web_search

results=[WebSearchResult(content="Holidays and Observances in Malaysia in 2026\n\nShowing: all Public holidaysPublic holidays and non-working daysHolidays and some observancesHolidays (incl. some local) and observancesHolidays (incl. all local) and observancesAll holidays/observances/religious eventsCustom – choose holidays... For: 2000200